# A trained 100-dimensional PINN as a certification benchmark

This notebook replaces the random target network by a standard physics-informed neural network for the manufactured Poisson problem

\[
-\Delta u=f\quad\text{in }\Omega=[-0.1,0.1]^{100},
\qquad u=g\quad\text{on }\partial\Omega.
\]

The exact solution is a nonconstant two-direction ridge function,

\[
u_*(x)=\sin(2.5\,a^\top x)+0.35\cos(1.75\,b^\top x),
\]

where $a,b\in\mathbb R^{100}$ are orthonormal dense directions. Hence

\[
f(x)=2.5^2\sin(2.5\,a^\top x)
+0.35\,1.75^2\cos(1.75\,b^\top x),
\qquad g=u_*|_{\partial\Omega}.
\]

The target architecture is exactly $100$-$50$-$50$-$50$-$1$ with tanh activations. The checkpoint was trained from the interior PDE residual and sampled Dirichlet boundary loss only; the exact solution is used for validation, not as supervised training data.

The half-width $0.1$ is an experimental scaling choice, not part of the Poisson equation itself. Because $a$ and $b$ are unit vectors, $a^\top x$ and $b^\top x$ can range on the order of one on this cube, so the two ridge modes remain nontrivial while the tanh preactivations stay in a range where a single-cell affine enclosure is still usable. On $[-1,1]^{100}$ the same frequencies would traverse much wider phase and preactivation intervals, making both ordinary PINN training and a global single-cell certificate substantially harder. The price of the small cube is $|\Omega|=0.2^{100}$, which is why raw norms are tiny and volume-normalized norms are also reported.

Certification compares interval arithmetic with the certified polynomial-Jacobian reductions Top-$k$, degree-capped Top-$k$, and coefficient-space PCA. The unreduced polynomial endpoint is structurally infeasible on the target architecture and is therefore not launched accidentally; reduced terms are always absorbed into a rigorously propagated pointwise remainder.

In [1]:
from __future__ import annotations

import math
import random
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch import nn

repo_root = Path.cwd()
while not (repo_root / 'src' / 'intervalnets').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from intervalnets import (
    IntervalTensor,
    PZIntegrationCell,
    enable_interval_eval,
    integrate_pz_onejet_squared,
    integrate_pz_value_squared,
    load_tanh_mlp_checkpoint,
    sequential_value_jacobian_laplacian,
)

torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
enable_interval_eval()

DIM = 100
HALF_WIDTH = 0.1
HIDDEN = (50, 50, 50)
SEED = 20260731
K1 = 2.5
K2 = 1.75
COS_AMPLITUDE = 0.35
CHECKPOINT = repo_root / 'notebooks' / 'checkpoints' / 'pinn_100d_poisson.pt'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'torch={torch.__version__}, dtype={torch.get_default_dtype()}, threads={torch.get_num_threads()}')
print(f'checkpoint={CHECKPOINT.relative_to(repo_root)}')

torch=2.13.0+cu130, dtype=torch.float64, threads=1
checkpoint=notebooks/checkpoints/pinn_100d_poisson.pt


## PDE, model, and efficient PINN residual

In [2]:
def dense_directions(dim=DIM):
    a = torch.ones(dim)
    a /= torch.linalg.vector_norm(a)
    b = torch.tensor([1.0 if i % 2 == 0 else -1.0 for i in range(dim)])
    b -= torch.dot(a, b) * a
    b /= torch.linalg.vector_norm(b)
    return a, b


A, B = dense_directions()


def exact_solution(x):
    return (torch.sin(K1 * (x @ A)) + COS_AMPLITUDE * torch.cos(K2 * (x @ B))).unsqueeze(-1)


def exact_gradient(x):
    s = x @ A
    t = x @ B
    return K1 * torch.cos(K1 * s).unsqueeze(-1) * A - COS_AMPLITUDE * K2 * torch.sin(K2 * t).unsqueeze(-1) * B


def forcing(x):
    return (K1**2 * torch.sin(K1 * (x @ A)) + COS_AMPLITUDE * K2**2 * torch.cos(K2 * (x @ B))).unsqueeze(-1)


def make_model():
    layers, previous = [], DIM
    for width in HIDDEN:
        layers.extend([nn.Linear(previous, width), nn.Tanh()])
        previous = width
    layers.append(nn.Linear(previous, 1))
    model = nn.Sequential(*layers)
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    return model


def sample_interior(n, generator):
    return (2.0 * torch.rand((n, DIM), generator=generator) - 1.0) * HALF_WIDTH


def sample_boundary(n, generator):
    x = sample_interior(n, generator)
    coordinate = torch.randint(DIM, (n,), generator=generator)
    sign = torch.where(torch.rand(n, generator=generator) < 0.5, -1.0, 1.0)
    x[torch.arange(n), coordinate] = HALF_WIDTH * sign
    return x


def pinn_residual(model, x):
    value, jacobian, laplacian = sequential_value_jacobian_laplacian(model, x)
    return value, jacobian, -laplacian - forcing(x)


model = make_model()
sum(parameter.numel() for parameter in model.parameters()), model

(10201, Sequential(
  (0): Linear(in_features=100, out_features=50, bias=True)
  (1): Tanh()
  (2): Linear(in_features=50, out_features=50, bias=True)
  (3): Tanh()
  (4): Linear(in_features=50, out_features=50, bias=True)
  (5): Tanh()
  (6): Linear(in_features=50, out_features=1, bias=True)
))

## Reproducible PINN training

Set `RETRAIN = True` to regenerate the checkpoint. The default loads the included deterministic checkpoint, so certification can be rerun in seconds. Training uses Adam with 512 fresh interior and 512 fresh boundary points per step and the ordinary loss

\[
\mathcal L(\theta)=\mathbb E_\Omega|{-\Delta u_\theta-f}|^2
+20\,\mathbb E_{\partial\Omega}|u_\theta-g|^2.
\]

The Laplacian helper propagates the exact Hessian trace through the tanh MLP and remains differentiable with respect to its parameters; it changes computational organization, not the PINN objective.

In [3]:
def train_pinn(model, steps=1500, batch_size=512, lr=2e-3):
    generator = torch.Generator().manual_seed(SEED + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=2e-4)
    history = []
    model.train()
    for step in range(1, steps + 1):
        interior = sample_interior(batch_size, generator)
        boundary = sample_boundary(batch_size, generator)
        _, _, residual = pinn_residual(model, interior)
        boundary_error = model(boundary) - exact_solution(boundary)
        residual_loss = residual.square().mean()
        boundary_loss = boundary_error.square().mean()
        loss = residual_loss + 20.0 * boundary_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        scheduler.step()
        if step == 1 or step % 100 == 0:
            history.append({
                'step': step,
                'loss': float(loss.detach()),
                'residual_loss': float(residual_loss.detach()),
                'boundary_loss': float(boundary_loss.detach()),
            })
    model.eval()
    return history


RETRAIN = False
if RETRAIN:
    training_history = train_pinn(model)
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'state_dict': model.state_dict(), 'seed': SEED}, CHECKPOINT)
else:
    model = load_tanh_mlp_checkpoint(CHECKPOINT)
    training_history = []

{'loaded_checkpoint': not RETRAIN, 'training_records': training_history[-3:]}

{'loaded_checkpoint': True, 'training_records': []}

## Candidate-network validation

In [4]:
validation_generator = torch.Generator().manual_seed(SEED + 222)
interior = sample_interior(8192, validation_generator)
boundary = sample_boundary(8192, validation_generator)
with torch.no_grad():
    prediction, network_jacobian, residual = pinn_residual(model, interior)
    target = exact_solution(interior)
    target_gradient = exact_gradient(interior)
    boundary_error = model(boundary) - exact_solution(boundary)
    error = prediction - target
    empirical_network_l2 = prediction.square().mean().sqrt()
    empirical_network_w12 = (prediction.square() + network_jacobian.square().sum(dim=(-2, -1), keepdim=True)).mean().sqrt()
    empirical_exact_l2 = target.square().mean().sqrt()
    empirical_exact_w12 = (target.square() + target_gradient.square().sum(dim=-1, keepdim=True)).mean().sqrt()

validation = {
    'solution_RMSE': float(error.square().mean().sqrt()),
    'solution_relative_L2_error': float(error.square().mean().sqrt() / target.square().mean().sqrt()),
    'solution_max_sample_error': float(error.abs().max()),
    'PDE_residual_RMSE': float(residual.square().mean().sqrt()),
    'boundary_RMSE': float(boundary_error.square().mean().sqrt()),
    'network_normalized_L2_MC': float(empirical_network_l2),
    'network_normalized_W12_MC': float(empirical_network_w12),
    'exact_normalized_L2_MC': float(empirical_exact_l2),
    'exact_normalized_W12_MC': float(empirical_exact_w12),
}
validation

{'solution_RMSE': 0.002859142422545456, 'solution_relative_L2_error': 0.007575039840436495, 'solution_max_sample_error': 0.030369810860190305, 'PDE_residual_RMSE': 0.01780581896203556, 'boundary_RMSE': 0.002867007950069476, 'network_normalized_L2_MC': 0.37731984665672036, 'network_normalized_W12_MC': 2.504706918197497, 'exact_normalized_L2_MC': 0.3774425590850363, 'exact_normalized_W12_MC': 2.5036437778283043}

## Certification diagnostics

The raw norm scales like $|\Omega|^{1/2}=0.2^{50}$, so both raw and volume-normalized intervals are reported. Volume normalization is not relative error: it only removes the factor $|\Omega|^{1/2}$. For an interval $[L,U]$, absolute width is $U-L$ and relative width is $(U-L)/\max(|L|,|U|)$ when the denominator is nonzero; the lower bound is not used as the denominator.

Before integration, the full certified Jacobian enclosure is summarized by mean component width, maximum component width, and the mean entrywise relative width

\[
\frac1N\sum_{ij}\frac{\overline J_{ij}-\underline J_{ij}}
{\max(|\underline J_{ij}|,|\overline J_{ij}|)},
\]

with exact-zero entries assigned zero. This relative width lies in $[0,2]$.

In [5]:
SQRT_VOLUME = (2.0 * HALF_WIDTH) ** (DIM / 2.0)
VOLUME = SQRT_VOLUME ** 2
DOMAIN = IntervalTensor.from_bounds([-HALF_WIDTH] * DIM, [HALF_WIDTH] * DIM)


def norm_interval(squared):
    return math.sqrt(max(0.0, float(squared.lower))), math.sqrt(max(0.0, float(squared.upper)))


def interval_metrics(bounds, prefix):
    lower, upper = map(float, bounds)
    width = upper - lower
    return {
        f'{prefix}_lower': lower,
        f'{prefix}_upper': upper,
        f'{prefix}_absolute_width': width,
        f'{prefix}_relative_width': width / max(abs(lower), abs(upper)) if max(abs(lower), abs(upper)) > 0.0 else 0.0,
        f'{prefix}_normalized_lower': lower / SQRT_VOLUME,
        f'{prefix}_normalized_upper': upper / SQRT_VOLUME,
        f'{prefix}_normalized_absolute_width': width / SQRT_VOLUME,
    }


def jacobian_width_metrics(enclosure):
    lower = torch.as_tensor(enclosure.lower)
    upper = torch.as_tensor(enclosure.upper)
    widths = upper - lower
    scales = torch.maximum(lower.abs(), upper.abs())
    relative_widths = torch.where(scales > 0.0, widths / scales, 0.0)
    return {
        'J_mean_component_width_before_integration': float(widths.mean()),
        'J_max_component_width_before_integration': float(widths.max()),
        'J_relative_mean_component_width_before_integration': float(relative_widths.mean()),
    }


def benchmark_polynomial(model, strategy='topk', **kwargs):
    cell = PZIntegrationCell.from_affine_box(DOMAIN)
    start = perf_counter()
    traced = model.eval_pz_onejet(
        cell.domain, return_trace=True, reduction_strategy=strategy, **kwargs
    )
    forward_s = perf_counter() - start
    jacobian_metrics = jacobian_width_metrics(traced.final.J.interval_enclosure())
    start = perf_counter()
    l2_integrated_pz = integrate_pz_value_squared(traced.final.Y, cell, output='pz')
    l2_squared = l2_integrated_pz.interval_enclosure()
    l2_integration_s = perf_counter() - start
    start = perf_counter()
    w12_integrated_pz = integrate_pz_onejet_squared(traced.final, cell, output='pz')
    w12_squared = w12_integrated_pz.interval_enclosure()
    w12_integration_s = perf_counter() - start
    return {
        'strategy': strategy,
        **kwargs,
        'forward_s': forward_s,
        'L2_integration_s': l2_integration_s,
        'W12_integration_s': w12_integration_s,
        'total_W12_s': forward_s + w12_integration_s,
        'J_terms': len(traced.final.J.terms),
        'J_degree': max(map(sum, traced.final.J.terms), default=0),
        'noise_count': traced.final.J.num_noise,
        'L2_integrated_PZ_terms': len(l2_integrated_pz.terms),
        'W12_integrated_PZ_terms': len(w12_integrated_pz.terms),
        'W12_integrated_PZ_noise': w12_integrated_pz.num_noise,
        **jacobian_metrics,
        **interval_metrics(norm_interval(l2_squared), 'L2'),
        **interval_metrics(norm_interval(w12_squared), 'W12'),
        'trace': traced.records,
    }


SQRT_VOLUME

1.1258999068426271e-35

## Interval and certified polynomial-reduction benchmarks

In [6]:
configurations = [
    ('topk-32', 'topk', dict(max_terms=32)),
    ('topk-64', 'topk', dict(max_terms=64)),
    ('topk-96', 'topk', dict(max_terms=96)),
    ('topk-128', 'topk', dict(max_terms=128)),
    ('topk-192', 'topk', dict(max_terms=192)),
    ('degree-64', 'degree', dict(max_terms=64, max_degree=2)),
    ('pca-64', 'pca', dict(max_terms=64, pca_rank=4, pca_candidates=32)),
]

polynomial_rows = []
for label, strategy, kwargs in configurations:
    row = benchmark_polynomial(model, strategy=strategy, **kwargs)
    row['method'] = label
    polynomial_rows.append(row)

start = perf_counter()
interval_w12 = model.sobolev_norm(DOMAIN, p=2.0, order=1, method='interval')
interval_total_s = perf_counter() - start
interval_l2 = model.lpnorm(DOMAIN, p=2.0, method='interval')
interval_jacobian = model.eval_jacobian(DOMAIN)
interval_row = {
    'method': 'interval',
    'total_W12_s': interval_total_s,
    **jacobian_width_metrics(interval_jacobian),
    **interval_metrics((interval_l2.lower, interval_l2.upper), 'L2'),
    **interval_metrics((interval_w12.lower, interval_w12.upper), 'W12'),
}

assert all(row['total_W12_s'] < 3.0 for row in polynomial_rows)
benchmark_rows = [interval_row, *polynomial_rows]

columns = [
    'method', 'total_W12_s',
    'L2_normalized_lower', 'L2_normalized_upper', 'L2_normalized_absolute_width', 'L2_relative_width',
    'W12_normalized_lower', 'W12_normalized_upper', 'W12_normalized_absolute_width', 'W12_relative_width',
    'J_mean_component_width_before_integration', 'J_max_component_width_before_integration',
    'J_relative_mean_component_width_before_integration', 'J_terms', 'J_degree',
]
[{key: row.get(key) for key in columns} for row in benchmark_rows]

[{'method': 'interval', 'total_W12_s': 0.18789246800042747, 'L2_normalized_lower': 0.0, 'L2_normalized_upper': 7.979522775780479, 'L2_normalized_absolute_width': 7.979522775780479, 'L2_relative_width': 1.0, 'W12_normalized_lower': 0.0, 'W12_normalized_upper': 88.8468047131762, 'W12_normalized_absolute_width': 88.8468047131762, 'W12_relative_width': 1.0, 'J_mean_component_width_before_integration': 17.548814954264703, 'J_max_component_width_before_integration': 20.8648129804914, 'J_relative_mean_component_width_before_integration': 1.9897599352068311, 'J_terms': None, 'J_degree': None}, {'method': 'topk-32', 'total_W12_s': 1.3675986349990126, 'L2_normalized_lower': 0.0, 'L2_normalized_upper': 2.6768306182215453, 'L2_normalized_absolute_width': 2.6768306182215453, 'L2_relative_width': 1.0, 'W12_normalized_lower': 0.0, 'W12_normalized_upper': 86.68683003796212, 'W12_normalized_absolute_width': 86.68683003796212, 'W12_relative_width': 1.0, 'J_mean_component_width_before_integration': 17.11

### Raw norm intervals and absolute widths

In [7]:
raw_columns = [
    'method',
    'L2_lower', 'L2_upper', 'L2_absolute_width', 'L2_relative_width',
    'W12_lower', 'W12_upper', 'W12_absolute_width', 'W12_relative_width',
]
[{key: row.get(key) for key in raw_columns} for row in benchmark_rows]

[{'method': 'interval', 'L2_lower': 0.0, 'L2_upper': 8.984143949899863e-35, 'L2_absolute_width': 8.984143949899863e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 1.0003260914983017e-33, 'W12_absolute_width': 1.0003260914983017e-33, 'W12_relative_width': 1.0}, {'method': 'topk-32', 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absolute_width': 3.0138433436891297e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 9.76006938642242e-34, 'W12_absolute_width': 9.76006938642242e-34, 'W12_relative_width': 1.0}, {'method': 'topk-64', 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absolute_width': 3.0138433436891297e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 9.683503828321607e-34, 'W12_absolute_width': 9.683503828321607e-34, 'W12_relative_width': 1.0}, {'method': 'topk-96', 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absolute_width': 3.0138433436891297e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 9.61

### Public default PZ APIs

In [8]:
start = perf_counter()
public_l2 = model.pz_l2norm(DOMAIN)
public_l2_s = perf_counter() - start
start = perf_counter()
public_w12 = model.pz_sobolev_norm(DOMAIN, order=1)
public_w12_s = perf_counter() - start

default_row = next(row for row in polynomial_rows if row['method'] == 'topk-96')
assert math.isclose(float(public_l2.upper), default_row['L2_upper'], rel_tol=1e-12)
assert math.isclose(float(public_w12.upper), default_row['W12_upper'], rel_tol=1e-12)
{
    'public_L2_s': public_l2_s,
    'public_W12_s': public_w12_s,
    **interval_metrics((public_l2.lower, public_l2.upper), 'public_L2'),
    **interval_metrics((public_w12.lower, public_w12.upper), 'public_W12'),
}

{'public_L2_s': 0.06607493099909334, 'public_W12_s': 2.057815976000711, 'public_L2_lower': -5e-324, 'public_L2_upper': 3.01384334368913e-35, 'public_L2_absolute_width': 3.01384334368913e-35, 'public_L2_relative_width': 1.0, 'public_L2_normalized_lower': -4.388184445513989e-289, 'public_L2_normalized_upper': 2.6768306182215458, 'public_L2_normalized_absolute_width': 2.6768306182215458, 'public_W12_lower': -5e-324, 'public_W12_upper': 9.611824696771708e-34, 'public_W12_absolute_width': 9.611824696771708e-34, 'public_W12_relative_width': 1.0, 'public_W12_normalized_lower': -4.388184445513989e-289, 'public_W12_normalized_upper': 85.37015269613308, 'public_W12_normalized_absolute_width': 85.37015269613308}

## Why the target unreduced polynomial is not executed

`reduction_strategy='none'` is a valid exact polynomial one-jet endpoint, but it is not a viable target-network benchmark. After the first tanh layer there are already roughly 100 domain-dependent derivative terms. The next chain-rule product couples these with about 150 derivative/value generators, producing on the order of $1.5\times10^4$ candidates; the third activation can then produce millions of candidates before canonicalization. Launching this path would violate the benchmark's memory and runtime purpose.

The unreduced endpoint remains covered by unit tests and by the small-network reference in `pz_w12_polynomial_reduction_benchmarks.ipynb`. Here, every target-network polynomial method is sound because the omitted tail is explicitly accumulated into a propagated pointwise remainder; no candidate term is simply dropped.

## What Top-$k$ reduction does

At each derivative-chain-rule product, every candidate monomial has a tensor coefficient $C_\alpha$. Top-$k$ scores it by its largest absolute component, keeps the $k$ highest-scoring exponent/coefficient pairs as dependent polynomial terms, and adds every discarded coefficient componentwise to a certified pointwise remainder radius. Equal exponent vectors are canonicalized before the final enclosure. Thus Top-$k$ is sound: it trades dependency information for a box remainder, but never deletes uncertainty. Larger $k$ preserves more correlation and cancellation, at the cost of more polynomial products and integration work.

PCA uses some of the discarded coefficient tensors differently: it retains a few shared coefficient-space directions and boxes only the orthogonal residual. This can preserve cancellation through later linear maps, although the benchmark below shows that the activation-approximation remainder, rather than the retained-support budget, dominates this particular PINN.

## Layer diagnostics for the default Top-96 method

For each hidden neuron, `tanh_prime_approximation_radius` is the certified coefficient $\delta_{\ell i}$ in the initial local enclosure

\[\tanh'(z_i)\in p_{\ell i}z_i+q_{\ell i}+\delta_{\ell i}[-1,1].\]

It is measured before multiplication by the incoming Jacobian and before any Top-$k$/PCA reduction, so it cleanly separates activation approximation error from compression error.

In [9]:
chosen = next(row for row in polynomial_rows if row['method'] == 'topk-96')
layer_diagnostics = [{
    'layer': record.layer_type,
    'seconds': record.elapsed_s,
    'Y_terms': record.summary['Y']['term_count'],
    'J_terms': record.summary['J']['term_count'],
    'J_degree': record.summary['J']['max_degree'],
    'J_remainder_mean_radius': record.summary['J']['remainder_mean_radius'],
    'J_remainder_max_radius': record.summary['J']['remainder_max_radius'],
} for record in chosen['trace']]
layer_diagnostics

[{'layer': 'Input', 'seconds': 0.0, 'Y_terms': 100, 'J_terms': 0, 'J_degree': 0, 'J_remainder_mean_radius': 0.0, 'J_remainder_max_radius': 0.0}, {'layer': 'Linear', 'seconds': 0.002622895999593311, 'Y_terms': 100, 'J_terms': 0, 'J_degree': 0, 'J_remainder_mean_radius': 0.0, 'J_remainder_max_radius': 0.0}, {'layer': 'Tanh', 'seconds': 0.02616049700009171, 'Y_terms': 150, 'J_terms': 96, 'J_degree': 1, 'J_remainder_mean_radius': 0.026446936553910286, 'J_remainder_max_radius': 0.07513935764656894}, {'layer': 'Linear', 'seconds': 0.009305661998951109, 'Y_terms': 150, 'J_terms': 96, 'J_degree': 1, 'J_remainder_mean_radius': 0.15294793951109945, 'J_remainder_max_radius': 0.23283848020502845}, {'layer': 'Tanh', 'seconds': 0.5622837949995301, 'Y_terms': 200, 'J_terms': 86, 'J_degree': 1, 'J_remainder_mean_radius': 0.17384077899371955, 'J_remainder_max_radius': 0.2799701365957077}, {'layer': 'Linear', 'seconds': 0.01211463400068169, 'Y_terms': 200, 'J_terms': 86, 'J_degree': 1, 'J_remainder_mean

### Initial per-neuron $\tanh'$ approximation-noise coefficients

In [ ]:
activation_records = [record for record in chosen['trace'] if record.layer_type == 'Tanh']
activation_error_columns = [
    record.summary['tanh_prime_approximation_radii'].detach().cpu().tolist()
    for record in activation_records
]
activation_error_summary = [{
    'hidden_layer': layer_index + 1,
    'min_delta': record.summary['tanh_prime_approximation_radius_min'],
    'mean_delta': record.summary['tanh_prime_approximation_radius_mean'],
    'max_delta': record.summary['tanh_prime_approximation_radius_max'],
} for layer_index, record in enumerate(activation_records)]
activation_error_rows = [{
    'neuron': neuron,
    **{f'hidden_layer_{layer + 1}_delta': values[neuron] for layer, values in enumerate(activation_error_columns)},
} for neuron in range(len(activation_error_columns[0]))]
activation_error_summary, activation_error_rows

## Glossary-conformant medium benchmark

This section is the canonical medium benchmark specified by
`docs/diagnostics_and_metrics_glossary.tex`. It executes two methods on the
same single-cell partition of $[-0.1,0.1]^{100}$:

1. `interval`: interval propagation and interval norm integration;
2. `affine_pz_topk96_symbolic`: affine polynomial-zonotope value and one-jet
   propagation with Top-96 reduction, followed by symbolic integration of the
   squared $L^2$ and $W^{1,2}$ quantities. The integrated scalar PZ is
   intervalized only after integration.

The implementation writes one self-contained canonical output directory per
method. Missing values use the literal `NA`; unimplemented Hessian and
certified PDE-residual quantities remain explicit with a non-`ok` status.
Squared norm intervals are retained, and the canonical relative norm width is
computed before the square root.

In [ ]:
import json
import subprocess
from datetime import datetime, timezone

import pandas as pd

from intervalnets import interval_forward

SCHEMA_VERSION = "1.2"
BENCHMARK_LEVEL = "medium"
PROBLEM_ID = "poisson_100d_ridge"
MODEL_ID = "pinn_100d_poisson_seed_20260731"
SPLIT_ID = "single_cell"
OUTPUT_ROOT = repo_root / "notebooks" / "benchmark_outputs" / "pinn_100d_poisson_medium"

METRICS_COLUMNS = [
    "schema_version", "benchmark_level", "problem_id", "model_id", "method_id",
    "run_id", "split_id", "quantity", "metric", "aggregation",
    "derivative_order", "layer", "neuron", "output_index", "input_index_a",
    "input_index_b", "cell_id", "value", "unit", "status",
]
CELL_INTERVAL_COLUMNS = [
    "run_id", "split_id", "cell_id", "cell_weight", "quantity", "output_index",
    "input_index_a", "input_index_b", "lower", "upper", "midpoint", "radius",
    "width", "magnitude", "mignitude", "local_relative_radius",
    "global_normalized_radius", "sign_certified", "status",
    "local_relative_width",
]
NORM_COLUMNS = [
    "run_id", "norm", "squared", "lower", "upper", "width", "relative_width",
    "value_contribution_upper", "gradient_contribution_upper",
    "hessian_contribution_upper", "value_contribution_width",
    "gradient_contribution_width", "hessian_contribution_width", "status",
    "domain_volume", "domain_volume_normalized_lower",
    "domain_volume_normalized_upper", "domain_volume_normalized_width",
]
ACTIVATION_COLUMNS = [
    "run_id", "split_id", "cell_id", "cell_weight", "layer", "neuron",
    "derivative_order", "preactivation_lower", "preactivation_upper",
    "preactivation_midpoint", "preactivation_radius", "preactivation_width",
    "approximation_kind", "approximation_error_radius",
    "approximation_error_diameter", "activation_scale",
    "normalized_approximation_radius", "noise_symbol_id", "shared_noise_group",
    "status",
]
COMPLEXITY_COLUMNS = [
    "run_id", "quantity", "n_alpha", "n_eta", "n_monomials",
    "n_mixed_monomials", "max_degree", "n_coefficients", "status",
]
TIMING_COLUMNS = ["run_id", "stage", "seconds", "status"]
SOUNDNESS_COLUMNS = [
    "run_id", "quantity", "sample_count", "failure_count", "max_violation",
    "invalid_interval_count", "nan_endpoint_count", "infinite_endpoint_count",
    "status",
]


def _tensor(value):
    return torch.as_tensor(value, dtype=torch.get_default_dtype()).detach().cpu()


def _nonnegative_squared_interval(enclosure):
    lower = max(0.0, float(enclosure.lower))
    upper = max(0.0, float(enclosure.upper))
    return lower, upper


def _outward_square(bounds):
    lower, upper = (max(0.0, float(v)) for v in bounds)
    return (
        float(np.nextafter(lower * lower, -np.inf)) if lower else 0.0,
        float(np.nextafter(upper * upper, np.inf)) if upper else 0.0,
    )


def _interval_trace(model, domain):
    current = domain
    records = []
    hidden_layer = 0
    for child in model:
        if isinstance(child, nn.Linear):
            current = interval_forward(child, current, enclosure_mode="box")
        elif isinstance(child, nn.Tanh):
            preactivation = current
            current = interval_forward(child, current, enclosure_mode="box")
            records.append({
                "layer": hidden_layer,
                "preactivation_lower": _tensor(preactivation.lower).reshape(-1),
                "preactivation_upper": _tensor(preactivation.upper).reshape(-1),
                "postactivation_lower": _tensor(current.lower).reshape(-1),
                "postactivation_upper": _tensor(current.upper).reshape(-1),
            })
            hidden_layer += 1
        else:
            current = interval_forward(child, current, enclosure_mode="box")
    return current, records


def _run_medium_methods(model):
    results = {}

    interval_start = perf_counter()
    interval_l2 = model.lpnorm(DOMAIN, p=2.0, method="interval")
    interval_l2_s = perf_counter() - interval_start
    interval_start = perf_counter()
    interval_w12 = model.sobolev_norm(DOMAIN, p=2.0, order=1, method="interval")
    interval_w12_s = perf_counter() - interval_start
    interval_start = perf_counter()
    interval_y, interval_activation = _interval_trace(model, DOMAIN)
    interval_j = model.eval_jacobian(DOMAIN)
    interval_enclosure_s = perf_counter() - interval_start
    results["interval"] = {
        "method_id": "interval",
        "run_id": "pinn100d_medium_interval",
        "Y": interval_y,
        "J": interval_j,
        "activation": interval_activation,
        "norms": {
            "L2": (float(interval_l2.lower), float(interval_l2.upper)),
            "L2_sq": _outward_square((interval_l2.lower, interval_l2.upper)),
            "W12": (float(interval_w12.lower), float(interval_w12.upper)),
            "W12_sq": _outward_square((interval_w12.lower, interval_w12.upper)),
        },
        "timings": {
            "L2_total": interval_l2_s,
            "W12_total": interval_w12_s,
            "final_enclosures": interval_enclosure_s,
        },
    }

    cell = PZIntegrationCell.from_affine_box(DOMAIN)
    pz_start = perf_counter()
    traced = model.eval_pz_onejet(
        cell.domain, return_trace=True, reduction_strategy="topk", max_terms=96,
        derivative_enclosure="affine", reduction_variant="A",
    )
    pz_forward_s = perf_counter() - pz_start
    pz_l2_start = perf_counter()
    pz_l2_integrated = integrate_pz_value_squared(traced.final.Y, cell, output="pz")
    pz_l2_sq = pz_l2_integrated.interval_enclosure()
    pz_l2_s = perf_counter() - pz_l2_start
    pz_w12_start = perf_counter()
    pz_w12_integrated = integrate_pz_onejet_squared(traced.final, cell, output="pz")
    pz_w12_sq = pz_w12_integrated.interval_enclosure()
    pz_w12_s = perf_counter() - pz_w12_start
    pz_activation = []
    hidden_layer = 0
    for record in traced.records:
        if record.layer_type != "Tanh":
            continue
        postactivation = record.value.interval_enclosure()
        pz_activation.append({
            "layer": hidden_layer,
            "preactivation_lower": record.summary["preactivation_lower"].detach().cpu(),
            "preactivation_upper": record.summary["preactivation_upper"].detach().cpu(),
            "postactivation_lower": _tensor(postactivation.lower).reshape(-1),
            "postactivation_upper": _tensor(postactivation.upper).reshape(-1),
            "rho0": record.summary["tanh_approximation_radii"].detach().cpu(),
            "rho1": record.summary["tanh_prime_approximation_radii"].detach().cpu(),
        })
        hidden_layer += 1
    pz_l2_sq_bounds = _nonnegative_squared_interval(pz_l2_sq)
    pz_w12_sq_bounds = _nonnegative_squared_interval(pz_w12_sq)
    results["affine_pz_topk96_symbolic"] = {
        "method_id": "affine_pz_topk96_symbolic",
        "run_id": "pinn100d_medium_affine_pz_topk96_symbolic",
        "Y": traced.final.Y.interval_enclosure(),
        "J": traced.final.J.interval_enclosure(),
        "Y_pz": traced.final.Y,
        "J_pz": traced.final.J,
        "activation": pz_activation,
        "trace": traced.records,
        "integrated_L2_pz": pz_l2_integrated,
        "integrated_W12_pz": pz_w12_integrated,
        "norms": {
            "L2_sq": pz_l2_sq_bounds,
            "L2": norm_interval(pz_l2_sq),
            "W12_sq": pz_w12_sq_bounds,
            "W12": norm_interval(pz_w12_sq),
        },
        "timings": {
            "onejet_construction": pz_forward_s,
            "L2_symbolic_integration": pz_l2_s,
            "W12_symbolic_integration": pz_w12_s,
            "W12_total": pz_forward_s + pz_w12_s,
        },
    }
    return results


medium_results = _run_medium_methods(model)

In [ ]:
def _interval_stats(lower, upper, scale):
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    midpoint = 0.5 * (lower + upper)
    radius = 0.5 * (upper - lower)
    width = upper - lower
    magnitude = np.maximum(np.abs(lower), np.abs(upper))
    mignitude = np.where((lower <= 0.0) & (upper >= 0.0), 0.0, np.minimum(np.abs(lower), np.abs(upper)))
    local = np.divide(radius, magnitude, out=np.zeros_like(radius), where=magnitude > 0.0)
    global_radius = np.divide(radius, scale, out=np.zeros_like(radius), where=scale > 0.0)
    return midpoint, radius, width, magnitude, mignitude, local, global_radius


def _family_metrics(base, method, quantity, lower, upper, derivative_order):
    lower = np.asarray(lower, dtype=float).reshape(-1)
    upper = np.asarray(upper, dtype=float).reshape(-1)
    scale = float(np.max(np.maximum(np.abs(lower), np.abs(upper)))) if lower.size else 0.0
    _, _, width, _, _, _, global_radius = _interval_stats(lower, upper, scale)
    specifications = [
        ("mean_width", "mean_weighted", float(np.mean(width))),
        ("max_width", "max", float(np.max(width))),
        ("q50_width", "q50", float(np.quantile(width, 0.50, method="linear"))),
        ("q90_width", "q90", float(np.quantile(width, 0.90, method="linear"))),
        ("q99_width", "q99", float(np.quantile(width, 0.99, method="linear"))),
        ("mean_global_normalized_radius", "mean_weighted", float(np.mean(global_radius))),
        ("max_global_normalized_radius", "max", float(np.max(global_radius))),
    ]
    rows = []
    for metric, aggregation, value in specifications:
        rows.append({**base, "quantity": quantity, "metric": metric,
                     "aggregation": aggregation, "derivative_order": derivative_order,
                     "value": value, "unit": "dimensionless", "status": "ok"})
    if quantity == "J":
        matrix_width = np.asarray(upper - lower, dtype=float)
        frobenius = float(np.linalg.norm(matrix_width.reshape(-1)))
        rows.extend([
            {**base, "quantity": "J", "metric": "mean_frobenius_width",
             "aggregation": "mean_weighted", "derivative_order": 1,
             "value": frobenius, "unit": "dimensionless", "status": "ok"},
            {**base, "quantity": "J", "metric": "max_frobenius_width",
             "aggregation": "max", "derivative_order": 1,
             "value": frobenius, "unit": "dimensionless", "status": "ok"},
        ])
    return rows


def _base_metric(result):
    return {
        "schema_version": SCHEMA_VERSION, "benchmark_level": BENCHMARK_LEVEL,
        "problem_id": PROBLEM_ID, "model_id": MODEL_ID,
        "method_id": result["method_id"], "run_id": result["run_id"],
        "split_id": SPLIT_ID, "layer": pd.NA, "neuron": pd.NA,
        "output_index": pd.NA, "input_index_a": pd.NA,
        "input_index_b": pd.NA, "cell_id": pd.NA,
    }


def _cell_interval_table(result):
    rows = []
    for quantity, enclosure in (("Y", result["Y"]), ("J", result["J"])):
        lower = np.asarray(_tensor(enclosure.lower), dtype=float)
        upper = np.asarray(_tensor(enclosure.upper), dtype=float)
        scale = float(np.max(np.maximum(np.abs(lower), np.abs(upper)))) if lower.size else 0.0
        midpoint, radius, width, magnitude, mignitude, local, global_radius = _interval_stats(lower, upper, scale)
        for index in np.ndindex(lower.shape):
            if quantity == "Y":
                output_index, input_a = (index[0] if index else 0), pd.NA
            else:
                output_index, input_a = index
            rows.append({
                "run_id": result["run_id"], "split_id": SPLIT_ID, "cell_id": 0,
                "cell_weight": 1.0, "quantity": quantity,
                "output_index": output_index, "input_index_a": input_a,
                "input_index_b": pd.NA, "lower": float(lower[index]),
                "upper": float(upper[index]), "midpoint": float(midpoint[index]),
                "radius": float(radius[index]), "width": float(width[index]),
                "magnitude": float(magnitude[index]), "mignitude": float(mignitude[index]),
                "local_relative_radius": float(local[index]),
                "global_normalized_radius": float(global_radius[index]),
                "sign_certified": int(not (lower[index] <= 0.0 <= upper[index])),
                "status": "ok",
                "local_relative_width": float(2.0 * local[index]),
            })
    return pd.DataFrame(rows, columns=CELL_INTERVAL_COLUMNS).sort_values(
        ["cell_id", "quantity", "output_index", "input_index_a", "input_index_b"],
        na_position="last", kind="stable", ignore_index=True,
    )


def _norm_table(result):
    rows = []
    for norm in ("L2", "W12"):
        for squared in (1, 0):
            key = norm + ("_sq" if squared else "")
            lower, upper = map(float, result["norms"][key])
            width = upper - lower
            volume_scale = VOLUME if squared else SQRT_VOLUME
            rows.append({
                "run_id": result["run_id"], "norm": norm, "squared": squared,
                "lower": lower, "upper": upper, "width": width,
                "relative_width": width / upper if upper > 0.0 else 0.0,
                "value_contribution_upper": pd.NA,
                "gradient_contribution_upper": pd.NA,
                "hessian_contribution_upper": pd.NA,
                "value_contribution_width": pd.NA,
                "gradient_contribution_width": pd.NA,
                "hessian_contribution_width": pd.NA,
                "status": "ok",
                "domain_volume": VOLUME,
                "domain_volume_normalized_lower": lower / volume_scale,
                "domain_volume_normalized_upper": upper / volume_scale,
                "domain_volume_normalized_width": width / volume_scale,
            })
    return pd.DataFrame(rows, columns=NORM_COLUMNS)


def _tanh_prime_hull(lower, upper):
    t_lo = np.tanh(lower)
    t_hi = np.tanh(upper)
    endpoint_lo = 1.0 - t_lo * t_lo
    endpoint_hi = 1.0 - t_hi * t_hi
    hull_lower = np.minimum(endpoint_lo, endpoint_hi)
    hull_upper = np.where((lower <= 0.0) & (upper >= 0.0), 1.0, np.maximum(endpoint_lo, endpoint_hi))
    return hull_lower, hull_upper


def _activation_tables(result):
    rows = []
    wide = {"neuron": np.arange(max(len(record["preactivation_lower"]) for record in result["activation"]))}
    for record in result["activation"]:
        layer = int(record["layer"])
        lower = np.asarray(record["preactivation_lower"], dtype=float)
        upper = np.asarray(record["preactivation_upper"], dtype=float)
        midpoint = 0.5 * (lower + upper)
        radius = 0.5 * (upper - lower)
        width = upper - lower
        y_lower = np.tanh(lower)
        y_upper = np.tanh(upper)
        d_lower, d_upper = _tanh_prime_hull(lower, upper)
        hulls = {0: (y_lower, y_upper), 1: (d_lower, d_upper)}
        post_lower = np.asarray(record["postactivation_lower"], dtype=float)
        post_upper = np.asarray(record["postactivation_upper"], dtype=float)
        y_scale = float(np.max(np.maximum(np.abs(post_lower), np.abs(post_upper))))
        y_normalized = (0.5 * (post_upper - post_lower) / y_scale) if y_scale > 0.0 else np.zeros_like(post_lower)
        padded = np.full(len(wide["neuron"]), np.nan)
        padded[:len(y_normalized)] = y_normalized
        wide[f"layer_{layer}"] = padded
        for derivative_order in (0, 1):
            hull_lower, hull_upper = hulls[derivative_order]
            scale = float(np.max(np.maximum(np.abs(hull_lower), np.abs(hull_upper))))
            if result["method_id"] == "interval":
                rho = 0.5 * (hull_upper - hull_lower)
                kind = "interval"
                noise_ids = [pd.NA] * len(lower)
            else:
                rho = np.asarray(record[f"rho{derivative_order}"], dtype=float)
                kind = "affine"
                noise_ids = [f"eta_l{layer}_n{neuron}_r{derivative_order}" for neuron in range(len(lower))]
            normalized = rho / scale if scale > 0.0 else np.zeros_like(rho)
            for neuron in range(len(lower)):
                rows.append({
                    "run_id": result["run_id"], "split_id": SPLIT_ID,
                    "cell_id": 0, "cell_weight": 1.0, "layer": layer,
                    "neuron": neuron, "derivative_order": derivative_order,
                    "preactivation_lower": lower[neuron],
                    "preactivation_upper": upper[neuron],
                    "preactivation_midpoint": midpoint[neuron],
                    "preactivation_radius": radius[neuron],
                    "preactivation_width": width[neuron],
                    "approximation_kind": kind,
                    "approximation_error_radius": rho[neuron],
                    "approximation_error_diameter": 2.0 * rho[neuron],
                    "activation_scale": scale,
                    "normalized_approximation_radius": normalized[neuron],
                    "noise_symbol_id": noise_ids[neuron],
                    "shared_noise_group": pd.NA, "status": "ok",
                })
    activation = pd.DataFrame(rows, columns=ACTIVATION_COLUMNS).sort_values(
        ["cell_id", "layer", "neuron", "derivative_order"], kind="stable", ignore_index=True,
    )
    wide_table = pd.DataFrame(wide)[["neuron"] + sorted([key for key in wide if key.startswith("layer_")])]
    return activation, wide_table


def _pz_complexity_row(result, quantity, pz):
    kinds = tuple(pz.noise_kinds)
    domain = {index for index, kind in enumerate(kinds) if kind == "domain"}
    approximation = set(range(len(kinds))) - domain
    support = list(pz.terms)
    mixed = sum(
        int(any(exp[index] for index in domain) and any(exp[index] for index in approximation))
        for exp in support
    )
    coefficient_dimension = int(np.prod(pz.shape)) if pz.shape else 1
    return {
        "run_id": result["run_id"], "quantity": quantity,
        "n_alpha": len(domain), "n_eta": len(approximation),
        "n_monomials": len(support), "n_mixed_monomials": mixed,
        "max_degree": max((sum(exp) for exp in support), default=0),
        "n_coefficients": coefficient_dimension * len(support), "status": "ok",
    }


def _complexity_table(result):
    if "Y_pz" in result:
        rows = [_pz_complexity_row(result, "Y", result["Y_pz"]),
                _pz_complexity_row(result, "J", result["J_pz"])]
    else:
        rows = [{"run_id": result["run_id"], "quantity": quantity, "status": "not_implemented"}
                for quantity in ("Y", "J")]
    rows.append({"run_id": result["run_id"], "quantity": "H", "status": "not_implemented"})
    return pd.DataFrame(rows).reindex(columns=COMPLEXITY_COLUMNS)


def _soundness_table(result):
    rows = []
    exact_values = {
        "Y": prediction.detach().cpu().reshape(-1, 1).numpy(),
        "J": network_jacobian.detach().cpu().reshape(-1, 1, DIM).numpy(),
    }
    for quantity, enclosure in (("Y", result["Y"]), ("J", result["J"])):
        lower = np.asarray(_tensor(enclosure.lower), dtype=float)
        upper = np.asarray(_tensor(enclosure.upper), dtype=float)
        values = exact_values[quantity]
        violation = np.maximum(np.maximum(lower - values, values - upper), 0.0)
        endpoints = np.concatenate([lower.reshape(-1), upper.reshape(-1)])
        rows.append({
            "run_id": result["run_id"], "quantity": quantity,
            "sample_count": values.shape[0],
            "failure_count": int(np.count_nonzero(np.any(violation > 0.0, axis=tuple(range(1, violation.ndim))))),
            "max_violation": float(np.max(violation)),
            "invalid_interval_count": int(np.count_nonzero(lower > upper)),
            "nan_endpoint_count": int(np.count_nonzero(np.isnan(endpoints))),
            "infinite_endpoint_count": int(np.count_nonzero(np.isinf(endpoints))),
            "status": "ok",
        })
    return pd.DataFrame(rows, columns=SOUNDNESS_COLUMNS)


def _metrics_table(result, cell_intervals, norms, complexity, timings, soundness):
    base = _base_metric(result)
    rows = []
    for quantity, derivative_order in (("Y", 0), ("J", 1)):
        family = cell_intervals[cell_intervals.quantity == quantity]
        rows.extend(_family_metrics(base, result["method_id"], quantity,
                                    family.lower, family.upper, derivative_order))
    for metric in ("mean_width", "max_width", "q50_width", "q90_width", "q99_width",
                   "mean_global_normalized_radius", "max_global_normalized_radius"):
        rows.append({**base, "quantity": "H", "metric": metric, "aggregation": "none",
                     "derivative_order": 2, "value": pd.NA, "unit": pd.NA,
                     "status": "not_implemented"})
    for _, row in norms.iterrows():
        quantity = row["norm"] + ("_sq" if row["squared"] else "")
        for metric in ("lower", "upper", "width", "relative_norm_width",
                       "domain_volume_normalized_lower",
                       "domain_volume_normalized_upper",
                       "domain_volume_normalized_width"):
            value = row["relative_width"] if metric == "relative_norm_width" else row[metric]
            rows.append({**base, "quantity": quantity, "metric": metric,
                         "aggregation": "none", "derivative_order": pd.NA,
                         "value": value, "unit": "dimensionless", "status": row["status"]})
    for residual_quantity, derivative_order in (("PDE_residual", 2), ("boundary_residual", 0), ("initial_residual", 0)):
        rows.append({**base, "quantity": residual_quantity, "metric": "linf_upper",
                     "aggregation": "max", "derivative_order": derivative_order, "value": pd.NA,
                     "unit": "dimensionless", "status": "not_implemented"})
    for _, row in complexity.iterrows():
        for column, metric in (("n_alpha", "n_alpha"), ("n_eta", "n_eta"),
                               ("n_monomials", "n_monomials"),
                               ("n_mixed_monomials", "n_mixed_monomials"),
                               ("max_degree", "max_degree")):
            rows.append({**base, "quantity": "complexity", "metric": metric,
                         "aggregation": "none", "derivative_order": pd.NA,
                         "value": row[column], "unit": "dimensionless", "status": row["status"],
                         "layer": pd.NA, "neuron": pd.NA})
    for _, row in timings.iterrows():
        rows.append({**base, "quantity": "runtime", "metric": "seconds",
                     "aggregation": row["stage"], "derivative_order": pd.NA,
                     "value": row["seconds"], "unit": "seconds", "status": row["status"]})
    rows.append({**base, "quantity": "memory", "metric": "bytes",
                 "aggregation": "peak", "derivative_order": pd.NA,
                 "value": pd.NA, "unit": "bytes", "status": "not_implemented"})
    for _, row in soundness.iterrows():
        for column, metric in (("failure_count", "failure_count"), ("max_violation", "max_violation")):
            rows.append({**base, "quantity": "soundness", "metric": metric,
                         "aggregation": row["quantity"], "derivative_order": pd.NA,
                         "value": row[column], "unit": "dimensionless", "status": row["status"]})
    return pd.DataFrame(rows).reindex(columns=METRICS_COLUMNS)

In [ ]:
medium_outputs = {}
try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None

for method_id, result in medium_results.items():
    method_dir = OUTPUT_ROOT / method_id
    method_dir.mkdir(parents=True, exist_ok=True)
    cell_intervals = _cell_interval_table(result)
    norms = _norm_table(result)
    activation, layer_radius_y = _activation_tables(result)
    complexity = _complexity_table(result)
    timings = pd.DataFrame([
        {"run_id": result["run_id"], "stage": stage, "seconds": seconds, "status": "ok"}
        for stage, seconds in result["timings"].items()
    ], columns=TIMING_COLUMNS)
    soundness = _soundness_table(result)
    metrics = _metrics_table(result, cell_intervals, norms, complexity, timings, soundness)

    assert list(metrics.columns) == METRICS_COLUMNS
    assert list(cell_intervals.columns) == CELL_INTERVAL_COLUMNS
    assert list(norms.columns) == NORM_COLUMNS
    assert list(activation.columns) == ACTIVATION_COLUMNS
    assert list(layer_radius_y.columns) == ["neuron", "layer_0", "layer_1", "layer_2"]
    assert list(complexity.columns) == COMPLEXITY_COLUMNS
    assert list(timings.columns) == TIMING_COLUMNS
    assert list(soundness.columns) == SOUNDNESS_COLUMNS
    assert activation.equals(activation.sort_values(
        ["cell_id", "layer", "neuron", "derivative_order"], kind="stable", ignore_index=True
    ))
    assert np.isclose(activation.cell_weight.groupby([activation.cell_id]).first().sum(), 1.0)
    assert not bool((norms.relative_width < 0.0).any() or (norms.relative_width > 1.0).any())
    assert int(soundness.failure_count.sum()) == 0
    assert np.allclose(
        cell_intervals.local_relative_width,
        2.0 * cell_intervals.local_relative_radius,
    )
    assert list(zip(norms["norm"], norms["squared"])) == [
        ("L2", 1), ("L2", 0), ("W12", 1), ("W12", 0),
    ]
    assert not norms[[
        "domain_volume", "domain_volume_normalized_lower",
        "domain_volume_normalized_upper", "domain_volume_normalized_width",
    ]].isna().any().any()
    norm_scales = np.where(norms.squared.astype(bool), VOLUME, SQRT_VOLUME)
    assert np.all(norms.domain_volume.to_numpy() == VOLUME)
    assert np.allclose(norms.domain_volume_normalized_lower, norms.lower / norm_scales)
    assert np.allclose(norms.domain_volume_normalized_upper, norms.upper / norm_scales)
    assert np.allclose(norms.domain_volume_normalized_width, norms.width / norm_scales)

    metadata = {
        "schema_version": SCHEMA_VERSION,
        "benchmark_level": BENCHMARK_LEVEL,
        "problem_id": PROBLEM_ID,
        "model_id": MODEL_ID,
        "method_id": method_id,
        "git_commit": git_commit,
        "dtype": str(torch.get_default_dtype()).replace("torch.", ""),
        "device": "cpu",
        "quantile_interpolation": "linear",
        "cell_average": "volume_weighted",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    (method_dir / "benchmark_metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
    )
    for filename, frame in {
        "metrics.csv": metrics,
        "cell_intervals.csv": cell_intervals,
        "norms.csv": norms,
        "complexity.csv": complexity,
        "timings.csv": timings,
        "soundness.csv": soundness,
        "activation_approximation.csv": activation,
        "layer_normalized_radius_Y.csv": layer_radius_y,
    }.items():
        frame.to_csv(method_dir / filename, index=False, na_rep="NA")
    medium_outputs[method_id] = {
        "metadata": metadata, "metrics": metrics, "cell_intervals": cell_intervals,
        "norms": norms, "complexity": complexity, "timings": timings,
        "soundness": soundness, "activation": activation,
        "layer_radius_Y": layer_radius_y,
    }

medium_norms_summary = pd.concat([
    output["norms"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)[[
    "method_id", "norm", "squared", "lower", "upper", "width", "relative_width",
    "domain_volume", "domain_volume_normalized_lower",
    "domain_volume_normalized_upper", "domain_volume_normalized_width", "status"
]]
medium_norms_summary

### Canonical norm table

Raw squared and unsquared intervals are shown together. Schema 1.2 stores
the physical domain volume and the mandatory domain-volume-normalized lower
endpoint, upper endpoint, and width directly in every canonical `norms.csv`
row. Squared rows are divided by $|\Omega|$ and unsquared rows by
$|\Omega|^{1/2}$.

In [ ]:
normalized_norm_view = medium_norms_summary[medium_norms_summary.squared == 0].copy()
normalized_norm_view[[
    "method_id", "norm", "domain_volume_normalized_lower",
    "domain_volume_normalized_upper", "domain_volume_normalized_width",
    "relative_width",
]]

### Final enclosure and complexity summary

In [ ]:
medium_enclosure_summary = pd.concat([
    output["metrics"].query(
        "quantity in ['Y', 'J'] and metric in ['mean_width', 'max_width', "
        "'mean_global_normalized_radius', 'max_global_normalized_radius', "
        "'mean_frobenius_width', 'max_frobenius_width']"
    )[["method_id", "quantity", "metric", "aggregation", "value", "status"]]
    for output in medium_outputs.values()
], ignore_index=True)
medium_enclosure_summary

In [ ]:
pd.concat([
    output["complexity"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)[["method_id"] + COMPLEXITY_COLUMNS]

### Per-neuron activation diagnostics

Each row below is canonical data from `activation_approximation.csv`. The
tables are ordered by hidden layer, neuron, and derivative order. The interval
method uses the constant interval-hull enclosure; the affine-PZ method reports
the certified approximation-error radius multiplying its fresh approximation
noise symbol.

In [ ]:
medium_activation_summary = pd.concat([
    output["activation"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)
medium_activation_layer_summary = (
    medium_activation_summary
    .groupby(["method_id", "layer", "derivative_order"], sort=True)
    .agg(
        preactivation_radius_mean=("preactivation_radius", "mean"),
        preactivation_radius_max=("preactivation_radius", "max"),
        approximation_error_radius_mean=("approximation_error_radius", "mean"),
        approximation_error_radius_max=("approximation_error_radius", "max"),
        normalized_approximation_radius_mean=("normalized_approximation_radius", "mean"),
        normalized_approximation_radius_max=("normalized_approximation_radius", "max"),
    )
    .reset_index()
)
medium_activation_layer_summary

In [ ]:
activation_views = {}
for method_id, output in medium_outputs.items():
    for derivative_order in (0, 1):
        view = output["activation"].query("derivative_order == @derivative_order")[
            ["layer", "neuron", "preactivation_lower", "preactivation_upper",
             "approximation_kind", "approximation_error_radius",
             "normalized_approximation_radius", "status"]
        ].reset_index(drop=True)
        activation_views[(method_id, derivative_order)] = view
        display(method_id, f"derivative_order={derivative_order}", view)

### Required layerwise normalized postactivation-radius tables

For each hidden layer, the denominator is the maximum magnitude of the
postactivation interval hull over all neurons in that layer. The row and
column indices are zero-based.

In [ ]:
for method_id, output in medium_outputs.items():
    display(method_id, output["layer_radius_Y"])

### Ranked worst activation enclosures

In [ ]:
largest_absolute_activation_radii = (
    medium_activation_summary
    .sort_values("approximation_error_radius", ascending=False, kind="stable")
    [["method_id", "layer", "neuron", "derivative_order", "preactivation_lower",
      "preactivation_upper", "approximation_kind", "approximation_error_radius",
      "normalized_approximation_radius"]]
    .head(20).reset_index(drop=True)
)
largest_normalized_activation_radii = (
    medium_activation_summary
    .sort_values("normalized_approximation_radius", ascending=False, kind="stable")
    [["method_id", "layer", "neuron", "derivative_order", "preactivation_lower",
      "preactivation_upper", "approximation_kind", "approximation_error_radius",
      "normalized_approximation_radius"]]
    .head(20).reset_index(drop=True)
)
largest_absolute_activation_radii, largest_normalized_activation_radii

### Validity and sampled-containment diagnostics

Sampling is only a diagnostic and is not presented as a proof of soundness.
Any nonzero failure count would, however, invalidate the corresponding
enclosure.

In [ ]:
pd.concat([
    output["soundness"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)[["method_id"] + SOUNDNESS_COLUMNS]

### Observed medium-benchmark result

The current implementation gives the following single-cell, domain-volume-normalized norm intervals:

| method | $L^2/|\Omega|^{1/2}$ | $W^{1,2}/|\Omega|^{1/2}$ |
|---|---:|---:|
| interval | $[0,7.979523]$ | $[0,88.846805]$ |
| affine PZ Top-96, symbolic integration | $[0,2.676831]$ | $[0,85.167723]$ |

The affine-PZ final function-value hull has width $5.415735$ versus $15.909816$ for interval propagation. The corresponding mean Jacobian-entry widths are $16.809482$ and $17.548815$.

Layerwise mean approximation-error radii $(\rho^{(0)},\rho^{(1)})$ for affine PZ are $(0.074179,0.275129)$, $(0.092422,0.305928)$, and $(0.133023,0.359351)$. For the constant interval-hull approximation they are $(0.741374,0.279480)$, $(0.999526,0.499543)$, and $(0.999971,0.499973)$. Thus the affine value enclosure yields a large improvement at every layer, while the affine $\tanh'$ enclosure becomes only marginally better than its interval hull as the zero-crossing bump widens.

Under schema 1.2 these values are no longer only a derived display: every squared and unsquared row in `norms.csv` stores the physical domain volume and its domain-volume-normalized lower endpoint, upper endpoint, and width. The squared normalized upper endpoints are $63.672784$ and $7893.754708$ for interval propagation, and $7.165422$ and $7253.541008$ for affine PZ. The complete per-neuron values, local relative widths, and familywise normalized radii are generated by the canonical cells above.

## Interpretation

- The checkpoint is a meaningful PDE candidate: sampled relative solution error is below one percent, while residual and boundary errors are independently reported.
- All reduced polynomial methods meet the three-second target on one CPU thread.
- The final raw norms are extremely small only because $|\Omega|^{1/2}=0.2^{50}$. Volume-normalized bounds should be compared with the Monte Carlo RMS norms, but they are not relative errors.
- Standard PINN training does **not** automatically yield a certification-friendly parameterization. The sampled normalized $W^{1,2}$ norm is modest, but all single-cell certified lower bounds are zero and the upper bounds are much larger. The per-neuron $\tanh'$ residual table shows sizeable initial derivative-approximation radii in every hidden layer, and the accumulated Jacobian remainder is then amplified by later linear maps. This explains why preserving more Top-$k$ terms or PCA directions yields only a small improvement.
- Squared $L^2$ and $W^{1,2}$ integration returns a scalar PZ. Pointwise residual uncertainty is converted to a fresh integrated global generator, and only this final PZ is intervalized before the square root.
- Top-192 is the tightest configuration that robustly remains below three seconds here: its normalized $W^{1,2}$ interval is $[0,85.0694]$, versus $[0,85.3702]$ for Top-96 and $[0,88.8468]$ for intervals. The gain from 96 to 192 terms is only about $0.35\%$, so support growth is already saturating.
- This is an application-level finding: improving certificate-aware training, activation enclosures, or domain decomposition is more important here than simply increasing the retained support.
- Degree-64 coincides with Top-64 because the retained terms are degree one. PCA-64 gives only a small improvement relative to its added runtime. These outcomes are reported rather than selected away.